## Scenario 1

**TICKET:** "Cron jobs haven't run in over an hour, and the customer says nothing in `/var/log` looks obviously broken."

**SYSTEM:** Ubuntu 22.04 LTS / WSL2

**START POINT:** You're logged in as `dennie`, sudo available. Nothing has been touched yet — go.

**Break down the input**

1. Cron jobs have not run in over an hour
2. Customer said nothing broken looks in `/var/log`

**Approach the issue**

1. Check `/var/log`, even though the customer said it's ok
2. Check and list current cron jobs

In [ ]:
dennie@HP:~$ ls -l /var/log
total 3048
-rw-r--r-- 1 root   root      2048 Sep 13 06:00 alternatives.log
-rw-r--r-- 1 root   root    197532 Sep 13 09:14 apt
drwxr-xr-x 2 root   root      4096 Sep 13 06:00 apt
-rw-r----- 1 root   adm       74821 Sep 13 09:20 auth.log
-rw-r--r-- 1 root   root       0    Sep  1 00:00 btmp
drwxr-xr-x 2 root   root      4096 Sep 13 06:00 cups
-rw-rw-r-- 1 root   utmp      2304 Sep 13 09:20 lastlog
drwxr-xr-x 2 root   root      4096 Sep 13 06:00 private
-rw-r----- 1 syslog adm      98234 Sep 13 09:20 syslog
drwxrwxr-x 2 syslog adm       4096 Sep 13 06:00 wtmp

No `cron.log` here — that's expected on modern Ubuntu, since cron logs through `journalctl` now, not a flat file. Nothing in this listing screams "broken" — permissions look normal, timestamps are current (things are actively being written to as of 09:20, a few minutes ago in this scenario).

In [ ]:
dennie@HP:~$ crontab -l
# Backup job - runs every hour on the hour
0 * * * * /home/dennie/scripts/backup.sh >> /home/dennie/backup.log 2>&1

Looks syntactically fine:
- every hour at minute 0
- **absolute path to the script**
- **output properly redirected**

According to this, it should have fired **at least once in the last hour**.

In [ ]:
sudo journalctl -u cron -n 100 --since "1 hour ago" -p err..crit --no-pager

In [ ]:
Sep 15 08:03:47 HP systemd[1]: cron.service: Main process exited, code=exited, status=1/FAILURE
Sep 15 08:03:47 HP systemd[1]: cron.service: Failed with result 'exit-code'.
Sep 15 08:03:47 HP systemd[1]: cron.service: Consumed 1.204s CPU time.

Here it is, the cron's service reports a failure with 1 code.

In [ ]:
systemctl status cron

In [ ]:
○ cron.service - Regular background program processing daemon
     Loaded: loaded (/usr/lib/systemd/system/cron.service; enabled; preset: enabled)
     Active: failed (Result: exit-code) since Sun 2026-09-15 08:03:47 UTC; 1h 38min ago
       Docs: man:cron(8)
    Process: 891 ExecStart=/usr/sbin/cron -f -P (code=exited, status=1/FAILURE)
   Main PID: 891 (code=exited, status=1/FAILURE)
        CPU: 1.204s

Confirmed: `Active: failed`, red/hollow `○` dot, matches the `journalctl` entries exactly — `enabled` is still true, meaning it would start on next boot, but right now it's simply dead and nothing is running it.

In [ ]:
cat /home/dennie/scripts/backup.sh

In [ ]:
#!/bin/bash
SRC="/home/dennie/linux-tut"
DEST="/home/dennie/backups/$(date +%Y%m%d).tar.gz"
mkdir -p /home/dennie/backups
tar -czf "$DEST" "$SRC"
echo "Backup completed: $DEST"

Script looks completely fine — absolute-ish paths, no obvious bug, nothing that would explain the failure on its own. Good due diligence to rule it out, but you've already found the actual root cause.

> **The  Root Cause**: cron itself is dead, so this script was never even invoked. Nothing here changes the diagnosis.

**The Fix**:

Fix the dead cron job, just run it manualy:

In [ ]:
sudo service cron status
sudo service cron start

**WSL-specific:** unlike real Linux (where cron is a systemd-managed service that auto-starts on boot), WSL's init doesn't always start cron automatically — you often need `sudo service cron start` after every `wsl --shutdown`/restart. Requires `sudo`.

Verify it's alive after:

In [ ]:
pgrep cron

If that returns a PID, it's running — your jobs should fire on the next scheduled minute.

In [ ]:
dennie@HP:~$ sudo service cron start
 * Starting periodic command scheduler cron                                                                   [ OK ]

In [ ]:
dennie@HP:~$ pgrep cron
14502

Or, use:

In [ ]:
sudo systemctl start cron
sudo systemctl enable cron

`enable` matters here since WSL "boots" are infrequent but you still want cron to survive them.

Cron is back up, new PID `14502` confirms it's actually running (not just reported OK). Nice choice using `service cron start` here too — that's the exact command called out in your WSL notes as the fallback some WSL setups need, and it works interchangeably with `systemctl start cron` under the hood (the `service` command is a legacy wrapper that dispatches to systemd when systemd is present).

One thing to close the loop on for a real ticket: would you also run `sudo systemctl enable cron` here, or is that unnecessary? And how would you _prove_ to the customer the fix actually worked (not just that the service is up)?
- yes i can use `systemctl enable cron` to Enable a service to start automatically on boot, and it looks like a good idea that it can run on each boot. `enable` matters here since WSL "boots" are infrequent but you still want cron to survive them. to prove the fix is ok i can run `systemctl status cron` of course, but i could go to `/var/log` to look for system logs about running cron job, right?

Good — `enable` reasoning is spot on. Small correction on the proof step though.

❌ "I could go to /var/log to look for system logs about running cron job"

✅ Use `journalctl -u cron` — that's the modern, service-scoped source of truth. You already found this out yourself in step 1 of this scenario: `ls -l /var/log` showed no `cron.log` file at all. Ubuntu 22.04+ routes cron's logging through `journald`, not a flat file in `/var/log`, so heading back to `/var/log` here would send you down the same dead end you already ruled out.

💡 Why: systemd-managed services log to the journal by default; `/var/log` flat files are the older syslog-era convention and often just don't exist anymore for journald-native services.

🔁 Support rule: **When you've already confirmed a log file doesn't exist for a service, don't revisit that path later in the same investigation — trust your own earlier findings and go straight to `journalctl -u <service>`.**

#### The actual proof sequence for this ticket:

In [ ]:
systemctl status cron          # confirms service is up right now
journalctl -u cron --since "5 minutes ago" --no-pager   # confirms it's actually firing new jobs post-fix

And the real closer — wait for the top of the next hour and check the backup itself landed:

In [ ]:
ls -l /home/dennie/backups/
cat /home/dennie/backup.log

#### Confirm the backup cron job is running

And the real closer — wait for the top of the next hour and check the backup itself landed:

In [ ]:
ls -l /home/dennie/backups/
cat /home/dennie/backup.log

That's the difference between "the service reports healthy" and "the thing the customer actually cares about is happening again."

**DEBRIEF — Scenario 1:**

- ✅ Correctly used `crontab -l` to rule out syntax issues before chasing the service itself
- ✅ Caught your own `-n 1` scoping mistake and self-corrected to add `-u cron.service`
- ✅ Systematic root-cause trace: crontab syntax → journalctl scoped to cron → systemctl status confirming failed state → script sanity check → fix → restart
- 🔁 One habit to reinforce: once you've ruled a location out (`/var/log` having no cron file), don't return to it later — carry that finding forward